# ENDF/B-VIII.1 decay data → `decay_lines.csv.gz` and `endf_states.csv`

`materials.isotopes.emissions` reads two tables built here from the ENDF decay sublibrary:

| table | one row per | columns |
|---|---|---|
| `decay_lines.csv.gz` | emission line | `Z, A, m, radiation, energy [eV], intensity [per decay]` |
| `endf_states.csv` | ENDF state (one decay file) | `Z, A, m, excitation [keV], half_life [s], nubase, match` |

`m` is the ENDF isomer number (LISO). The `nubase` column names the NUBASE2020 state (`nubase.ipynb`) that each
ENDF state is: the two number isomers differently, so a NUBASE isomer is found through this column.

**Source.** The ENDF/B-VIII.1 decay sublibrary (here the FISPACT-II package's `decay` directory); set
`DECAY_DIR` below. Parsed with openmc (`openmc.data.Evaluation`, `openmc.data.Decay`).
Run the notebook from this directory; it overwrites the two tables in `../materials/library`.

In [1]:
import gzip
import math
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from openmc.data import Decay, Evaluation

DECAY_DIR = Path(os.environ.get("ENDF_DECAY_DIR",
                                "~/nuclear_data/ENDFB81data/decay")).expanduser()
LIBRARY = Path("../materials/library")
EVALUATION = "ENDF/B-VIII.1"
warnings.filterwarnings("ignore", message="Using UFloat objects with std_dev==0")

## 1. Read every decay file

openmc gives, per file: the state (`target`: Z, A, LISO and its excitation energy ELIS), the half-life, and the
discrete spectra of MF8/MT457 (energies and intensities per decay, times the discrete normalization).

In [2]:
def nominal(x) -> float:
    return float(getattr(x, "nominal_value", x))


decays = []
for path in sorted(p for p in DECAY_DIR.iterdir() if p.is_file()):
    evaluation = Evaluation(path)
    t = evaluation.target
    decays.append(((t["atomic_number"], t["mass_number"], t["isomeric_state"]), t["excitation_energy"],
                   Decay(evaluation)))
print(len(decays), "decay files")

3820 decay files


## 2. Lines

| radiation | energy | intensity |
|---|---|---|
| `gamma`, `xray` | photon energy | photons per decay |
| `annihilation` | 511 keV | photons per decay: ENDF lists them among the X-rays at 511 keV; relabelled |
| `beta-` | endpoint energy of the branch | branch intensity |
| `beta+` | positron endpoint = transition energy − 2 m_e c² (ENDF gives the EC/β+ transition energy) | positrons per decay (electron capture excluded) |
| `alpha` | alpha energy | alphas per decay |

In [3]:
TWO_ME, HALF_ME = 1.02199895e6, 0.510998950e6     # eV
KINDS = {"gamma": "gamma", "xray": "xray", "beta-": "beta-", "alpha": "alpha"}   # openmc's spectrum names


def lines_of(decay: Decay):
    """(radiation, energy [eV], intensity per decay) of every discrete line."""
    for key, spectrum in decay.spectra.items():
        norm = nominal(spectrum.get("discrete_normalization", 1.0))
        for line in spectrum.get("discrete") or []:
            energy = nominal(line["energy"])
            if key in KINDS:
                kind = "annihilation" if key == "xray" and abs(energy - HALF_ME) < 10.0 else KINDS[key]
                yield kind, energy, norm * nominal(line["intensity"])
            elif key == "ec/beta+":
                positrons = norm * nominal(line.get("positron_intensity", 0.0))
                if positrons > 0:
                    yield "beta+", energy - TWO_ME, positrons


lines = pd.DataFrame([(z, a, m, kind, energy, intensity)
                      for (z, a, m), _, decay in decays
                      for kind, energy, intensity in lines_of(decay) if intensity > 0],
                     columns=["Z", "A", "m", "radiation", "energy", "intensity"])
lines.radiation.value_counts()

radiation
gamma           65544
xray            20035
beta-           10241
beta+            5635
alpha            1482
annihilation      499
Name: count, dtype: int64

In [4]:
lines[(lines.Z == 27) & (lines.A == 60) & (lines.m == 0) & (lines.intensity > 1e-3)]   # Co-60

,Z,A,m,radiation,energy,intensity
20406,27,60,0,gamma,1173228.0,0.998500
20407,27,60,0,gamma,1332492.0,0.999826
20410,27,60,0,beta-,318151.8,0.998800
20411,27,60,0,beta-,1491392.0,0.001200


## 3. States, and which NUBASE state each one is

ENDF numbers only the states that have a decay file; NUBASE letters every level, short-lived ones included, so
the m of the two differs for many isomers (Ag-110: NUBASE `Ag110m` is a 0.66 µs level, `Ag110n` the 250 d isomer,
which is ENDF's m = 1). As in `onegroupxs` (`DecayLibrary.isomer`), the excitation energy is the quantity both
share, and the half-life decides when there is no energy:

1. **ground state** (m = 0): NUBASE's ground state.
2. **energy**: the NUBASE excited state of the same Z, A whose excitation is within max(2 keV, 1 %) of ELIS
   (the nearest if several).
3. **half-life**: when ENDF gives no excitation (ELIS = 0) or none is close, the one NUBASE excited state whose
   half-life is within 25 %.
4. **none**: otherwise, mostly sub-second states; they have lines but no NUBASE name.
5. **ambiguous**: two ENDF states would take the same NUBASE state by half-life (Eu-136 m1 and m2, 3.3 and
   3.8 s, ELIS unknown, against NUBASE's one Eu136m): neither gets it.

In [5]:
nubase = pd.read_csv(LIBRARY / "nubase2020.csv")
is_ground = nubase.ZAID == 1000 * nubase.Z + nubase.A
excited = nubase[~is_ground]
ground = nubase[is_ground].set_index(["Z", "A"]).symbol

ATOL, RTOL, HALF_LIFE_RTOL = 2.0, 0.01, 0.25     # keV, -, -


def nubase_state(z: int, a: int, m: int, elis: float, half_life: float) -> tuple[str, str]:
    """(NUBASE symbol, how it was matched) of an ENDF state; ('', 'none') if no match."""
    if m == 0:
        return ground.get((z, a), ""), "ground" if (z, a) in ground.index else "none"
    candidates = excited[(excited.Z == z) & (excited.A == a)]
    if elis > 0:
        distance = (candidates.excitation - elis).abs()
        close = candidates[distance <= max(ATOL, RTOL * elis)]
        if len(close):
            return close.loc[distance[close.index].idxmin(), "symbol"], "energy"
    if 0 < half_life < math.inf:
        ratio = np.abs(np.log(candidates.half_life / half_life))
        close = candidates[ratio < math.log(1 + HALF_LIFE_RTOL)]
        if len(close) == 1:
            return close.symbol.iloc[0], "half-life"
    return "", "none"


rows = []
for (z, a, m), elis, decay in decays:
    half_life = nominal(decay.half_life)
    symbol, how = nubase_state(z, a, m, elis / 1e3, half_life)
    rows.append((z, a, m, elis / 1e3, half_life, symbol, how))
states = pd.DataFrame(rows, columns=["Z", "A", "m", "excitation", "half_life", "nubase", "match"])

# one NUBASE state per ENDF state: when several claim one, an energy match wins; otherwise none keeps it
claimed = states[states.nubase != ""].groupby("nubase")
for symbol, group in claimed:
    if len(group) > 1:
        losers = group.index if (group.match != "energy").all() else group.index[group.match != "energy"]
        states.loc[losers, ["nubase", "match"]] = ["", "ambiguous"]
states.match.value_counts()

match
ground       3082
energy        488
half-life     192
none           56
ambiguous       2
Name: count, dtype: int64

How many ENDF isomers have a different number in NUBASE (a NUBASE letter other than the m-th of m, n, p, ...),
and examples:

In [6]:
isomers = states[(states.m > 0) & states.match.isin(["energy", "half-life"])]
nubase_index = nubase.set_index("symbol").ZAID % 10
renumbered = isomers[isomers.nubase.map(nubase_index) != isomers.m]
print(len(renumbered), "of", len(isomers), "matched isomers are numbered differently")
renumbered.head(10)

51 of 680 matched isomers are numbered differently


,Z,A,m,excitation,half_life,nubase,match
70,47,110,1,117.59,2.157926e+07,Ag110n,energy
86,47,118,1,127.74,2.000000e+00,Ag118n,energy
90,47,120,1,203.00,3.200000e-01,Ag120n,energy
214,85,196,1,157.90,1.100000e-05,At196n,energy
301,79,198,1,811.70,1.963008e+05,Au198n,energy
402,83,188,1,0.00,2.650000e-01,Bi188n,half-life
706,96,244,1,0.00,5.000000e-07,Cm244n,half-life
791,55,122,1,127.10,3.600000e-01,Cs122p,energy
913,66,157,1,199.50,2.160000e-02,Dy157n,energy
1024,63,152,2,147.81,5.760000e+03,Eu152r,energy


In [7]:
states[(states.m > 0) & (states.match == "none")].head(10)   # no NUBASE state: mostly sub-second

,Z,A,m,excitation,half_life,nubase,match
38,47,94,2,6670.0,0.4000,,none
88,47,119,1,0.0,6.0000,,none
210,85,194,1,0.0,0.2500,,none
387,107,262,1,0.0,0.0220,,none
395,83,184,1,0.0,0.0066,,none
414,83,194,1,0.0,125.0000,,none
415,83,194,2,0.0,115.0000,,none
742,27,70,1,0.0,0.5000,,none
823,55,144,1,0.0,1.0000,,none
859,29,76,1,0.0,1.2700,,none


## 4. Checks

In [8]:
def row(z, a, m):
    return states[(states.Z == z) & (states.A == a) & (states.m == m)].iloc[0]


assert row(47, 110, 1).nubase == "Ag110n"                    # 250 d Ag-110m is NUBASE's second state
assert row(11, 24, 1).nubase == "Na24m" and row(11, 24, 1).match == "energy"
assert row(27, 60, 0).nubase == "Co60"
assert not states.duplicated(["Z", "A", "m"]).any()
matched = states[states.nubase != ""]
assert not matched.nubase.duplicated().any(), matched[matched.nubase.duplicated(keep=False)]
co60 = lines[(lines.Z == 27) & (lines.A == 60) & (lines.m == 0) & (lines.radiation == "gamma")]
assert abs(co60.intensity[(co60.energy - 1332.5e3).abs() < 1e3].sum() - 0.9998) < 1e-3

## 5. Write

In [9]:
with gzip.open(LIBRARY / "decay_lines.csv.gz", "wt") as fh:
    fh.write(f"# {EVALUATION} decay data, discrete lines; energy [eV], intensity per decay\n")
    lines.to_csv(fh, index=False, float_format="%.6g")
states.to_csv(LIBRARY / "endf_states.csv", index=False, float_format="%.6g")
print(len(lines), "lines,", len(states), "states written to", LIBRARY)

103436 lines, 3820 states written to ../materials/library
